# Mobile App for Lottery Addiction

Many people start playing the lottery for fun, but for some this activity turns into a habit that escalates into addiction. A medical institute that aims to prevent and treat gambling addiction wants to build a dedicated mobile app to help lottery addicts better estimate their chances of winning.

In this project, we write the logic behind the app: a set of functions that answer questions like:

- What is the probability of winning the big prize with a single ticket?
- What is the probability of winning the big prize if we play 40 different tickets (or any other number)?
- What is the probability of having at least five (or four, three, two) winning numbers on a single ticket?

We focus on the [6/49 lottery](https://en.wikipedia.org/wiki/Lotto_6/49) and use historical data from the national 6/49 lottery game in Canada, which covers 3,665 drawings from 1982 to 2018.

## Core Functions

We'll need to calculate probabilities repeatedly, so we start by writing two functions:

- `factorial()` — calculates factorials.
- `combinations()` — calculates the number of combinations when we take `k` objects from a group of `n` objects.

In [ ]:
def factorial(n):
    final_product = 1
    for i in range(n, 0, -1):
        final_product *= i
    return final_product

def combinations(n, k):
    numerator = factorial(n)
    denominator = factorial(k) * factorial(n - k)
    return numerator / denominator

## One-ticket Probability

In the 6/49 lottery, six numbers are drawn from a set of 49 numbers (1 to 49), without replacement. A player wins the big prize if the six numbers on their ticket match all six numbers drawn — the order doesn't matter.

For a single ticket there is exactly one successful outcome, and the total number of possible outcomes is the number of six-number combinations we can draw from 49 numbers. The function below takes the user's six numbers and prints the probability of winning the big prize in a way that's easy to understand.

In [ ]:
def one_ticket_probability(user_numbers):
    n_combinations = combinations(49, 6)
    probability_one_ticket = 1 / n_combinations
    percentage_form = probability_one_ticket * 100

    print('''Your chances to win the big prize with the numbers {} are {:.7f}%.
In other words, you have a 1 in {:,} chances to win.'''.format(user_numbers, percentage_form, int(n_combinations)))

In [ ]:
one_ticket_probability([2, 43, 22, 23, 11, 5])
print()
one_ticket_probability([9, 26, 41, 7, 15, 6])

Whatever six numbers the user picks, the probability is the same: there are 13,983,816 possible combinations, so a single ticket has a 1 in 13,983,816 chance of winning the big prize.

## Historical Data Check for Canada Lottery

The institute also wants users to be able to compare their ticket against the historical lottery data in Canada and see whether they would have ever won by now. We start by exploring the data set.

In [ ]:
import pandas as pd

lottery_canada = pd.read_csv('649.csv')
print(lottery_canada.shape)

In [ ]:
lottery_canada.head(3)

In [ ]:
lottery_canada.tail(3)

Each row describes a single drawing. The six winning numbers are stored in the columns `NUMBER DRAWN 1` to `NUMBER DRAWN 6`, and there's also a `BONUS NUMBER` that we won't need for the big prize.

## Function for Historical Data Check

We now write a function that prints:

- the number of times the combination selected by the user occurred in the Canada data set, and
- the probability of winning the big prize in the next drawing with that combination.

First, `extract_numbers()` returns the six winning numbers of a drawing as a Python set. Sets are unordered, which matches the rules of the lottery — `{3, 11, 12, 14, 41, 43}` and `{43, 41, 14, 12, 11, 3}` are the same ticket.

In [ ]:
def extract_numbers(row):
    row = row[4:10]
    row = set(row.values)
    return row

winning_numbers = lottery_canada.apply(extract_numbers, axis=1)
winning_numbers.head()

Then, `check_historical_occurence()` converts the user's numbers to a set and compares it against every set in `winning_numbers`. The comparison returns a Series of Boolean values, and summing it gives the number of matches.

In [ ]:
def check_historical_occurence(user_numbers, historical_numbers):
    user_numbers_set = set(user_numbers)
    check_occurrence = historical_numbers == user_numbers_set
    n_occurrences = check_occurrence.sum()

    if n_occurrences == 0:
        print('''The combination {} has never occurred.
This doesn't mean it's more likely to occur now. Your chances to win the big prize in the next drawing using the combination {} are 0.0000072%.
In other words, you have a 1 in 13,983,816 chances to win.'''.format(user_numbers, user_numbers))
    else:
        print('''The number of times combination {} has occurred in the past is {}.
Your chances to win the big prize in the next drawing using the combination {} are 0.0000072%.
In other words, you have a 1 in 13,983,816 chances to win.'''.format(user_numbers, n_occurrences, user_numbers))

In [ ]:
check_historical_occurence([33, 36, 37, 39, 8, 41], winning_numbers)
print()
check_historical_occurence([3, 2, 1, 4, 5, 6], winning_numbers)

The first test combination is the second drawing in the data set, so it shows up in the historical data. The important point for the user is the second part of each message: past results don't change the odds. Every drawing is independent, so any combination — one that already came up or one that never did — has the same 1 in 13,983,816 chance of winning the next drawing.

## Multi-ticket Probability

Lottery addicts usually play more than one ticket in a single drawing, thinking this significantly increases their chances of winning. The function below takes the number of different tickets played and prints the probability of winning the big prize.

In [ ]:
def multi_ticket_probability(n_tickets):
    n_combinations = combinations(49, 6)
    probability = n_tickets / n_combinations
    percentage_form = probability * 100

    if n_tickets == 1:
        print('''Your chances to win the big prize with one ticket are {:.6f}%.
In other words, you have a 1 in {:,} chances to win.'''.format(percentage_form, int(n_combinations)))
    else:
        combinations_simplified = round(n_combinations / n_tickets)
        print('''Your chances to win the big prize with {:,} different tickets are {:.6f}%.
In other words, you have a 1 in {:,} chances to win.'''.format(n_tickets, percentage_form, combinations_simplified))

In [ ]:
test_inputs = [1, 10, 100, 10000, 1000000, 6991908, 13983816]

for test_input in test_inputs:
    multi_ticket_probability(test_input)
    print('------------------------')

Even with 10,000 different tickets, the chance of winning the big prize is only about 0.072%. A player needs 6,991,908 different tickets for a 50% chance, and all 13,983,816 possible tickets to be certain of winning.

## Less Winning Numbers — Function

Most 6/49 lotteries also have smaller prizes for tickets that match two, three, four, or five of the six numbers drawn. The function below takes an integer between 2 and 5 and prints the probability of having **exactly** that many winning numbers on a single ticket.

For a ticket to match exactly `n` numbers:

- `n` of the ticket's six numbers must be among the six winning numbers: `combinations(6, n)` ways.
- The other `6 - n` numbers must come from the 43 numbers that weren't drawn: `combinations(43, 6 - n)` ways.

Multiplying the two gives the number of successful outcomes, and the total number of outcomes is still the number of six-number combinations.

In [ ]:
def probability_less_6(n_winning_numbers):
    n_combinations_ticket = combinations(6, n_winning_numbers)
    n_combinations_remaining = combinations(43, 6 - n_winning_numbers)
    successful_outcomes = n_combinations_ticket * n_combinations_remaining

    n_combinations_total = combinations(49, 6)
    probability = successful_outcomes / n_combinations_total
    probability_percentage = probability * 100

    combinations_simplified = round(n_combinations_total / successful_outcomes)
    print('''Your chances of having {} winning numbers with this ticket are {:.6f}%.
In other words, you have a 1 in {:,} chances to win.'''.format(n_winning_numbers, probability_percentage, int(combinations_simplified)))

In [ ]:
for test_input in [2, 3, 4, 5]:
    probability_less_6(test_input)
    print('--------------------------')

Matching exactly five numbers happens about once every 54,201 tickets, and exactly four numbers about once every 1,032 tickets. Only the smallest prizes are reasonably likely: exactly three numbers is about 1 in 57, and exactly two numbers is about 1 in 8.

## Conclusion

In this project, we wrote the logic for the app. Given a ticket or a number of tickets, it can calculate:

- The probability of winning the big prize with a single ticket.
- The number of times a combination occurred in the Canada lottery data set, together with the probability of winning the big prize in the next drawing with that combination.
- The probability of winning the big prize with multiple tickets.
- The probability of having exactly two, three, four, or five winning numbers.

Possible next steps:

- Make the output of `probability_less_6()` easier to read by adding more context.
- Write a function that calculates the probability of having **at least** two, three, four, or five winning numbers (for example, at least four = exactly four + exactly five + exactly six).
- Combine the one-ticket and historical data check functions into a single function.